In [ ]:
from synergy_dataset import Dataset, iter_datasets

In [ ]:
import psycopg2
import pandas as pd
con = psycopg2.connect(
    dbname="oa",
    user="username",
    password="password",
    host="host",
    port="port"
)

for d in iter_datasets():
    #if d.name != "Walker_2018":  
    #    continue
    #elif d.name == "Walker_2018":
    #    break
    df = d.to_frame()
    with con:
        vals = ",".join([f"'{x.split('/')[-1]}'" for x in df.index.values])
        # Get work-author doublets
        stmt = f"SELECT * FROM openalex.works WHERE work_id in ({vals})"
        wa = pd.read_sql(stmt, con)
    df['work_id'] = [x.split('/')[-1].replace("'", "") for x in df.index.values]
    df = pd.merge(df,wa[['work_id','publication_year']],on='work_id',how='left')
    print(f"found publication year for {round(100*df['publication_year'].value_counts().sum()/df.shape[0],2)}% records")
    df.to_csv(f"{d.name}.csv", index=False)
    print(f"saved {d.name} as a csv file")